In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class RopelessMLA(nn.Module):
    def __init__(self, d_model, n_heads, kv_latent_dim):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads  # dimension per head
    
        # Projection layers
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.Wdkv = nn.Linear(d_model, kv_latent_dim, bias=False)
        self.W_uk = nn.Linear(kv_latent_dim, d_model, bias=False)
        self.W_uv = nn.Linear(kv_latent_dim, d_model, bias=False)
        self.W_o = nn.Linear(kv_latent_dim, d_model, bias=False)

        self.ln = nn.LayerNorm(kv_latent_dim)
        self.register_buffer('absorbed_k', None) # Holds W_q @ W_uk

    def forward(self, x, kv_cache=None, past_length=0):
        B, S, D = x.size()

        # Compute absorbed_k once: W_q @ W_uk, shape:(D, latent_dim)
        if self.absorbed_k is None:
            absorbed = torch.matmul(self.W_q.weight, self.W_uk.weight) #(D, latent_dim)
            self.absorbed_k = absorbed.view(self.n_heads, self.head_dim, -1) #(n_heads, head_dim, latent_dim)

        # compress x into Ckv
        self.c_kv = self.ln(self.W_dkv(x)) #(B, S, latent_dim)
        if kv_cache is None:
            c_kv = new_c_kv
        else:
            # appended
            c_kv = torch.cat([kv_cache, new_c_kv], dim=1) #(B, S_total, latent_dim)

        s_full = c_kv.size(1)

        

        